# 实验报告 03：正则表达式与网络结构化数据

**课程实验**：Python 基础与数据处理　　**实验编号**：EXP03
**学号**：20244081132　　**姓名**：唐林

> **提交要求对照**
> 1. 已在独立 conda 虚拟环境 `exp03`（非 base）中完成，环境内含 Python 3 / Jupyter / 正则与 BeautifulSoup 所需的全部库；
> 2. 第一个代码框填入本人学号，其后按代码单元顺序运行；
> 3. 每个任务的关键输出均已显示，并用 Markdown 说明匹配规则与解析结果；
> 4. **重启内核并运行全部代码后**输出已嵌入；文件命名为 `20244081132_唐林_exp03.ipynb`。

---

## 目录
- 1.1 身份与环境记录
- 实验 1：正则表达式的基础匹配（转义字符 / 择一 / 量词）
- 实验 2：分组、回溯引用与替换
- 实验 3：使用 BeautifulSoup 解析 HTML

## 1.1 第一个代码框：身份与环境记录

本代码框按题目要求完成身份绑定：写入学号，读取当前 conda 环境名，并用 `platform` / `uuid` / `hashlib` 生成机器指纹，组合成唯一 `RUN_ID`。同时在本框导入实验所需的标准库 `re` 与第三方库 `bs4`。

In [1]:
import hashlib
import os
import platform
import uuid
import sys

import re
from bs4 import BeautifulSoup

# 修改为自己的学号
STUDENT_ID = 20244081132
STUDENT_NAME = "唐林"

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[姓名]       {STUDENT_NAME}")
print(f"[机器指纹]   {MACHINE_FP}")
print(f"[RUN_ID]     {RUN_ID}")
print(f"[Python]     {sys.version.split()[0]}")

[Conda 环境] exp03
[学号]       20244081132
[姓名]       唐林
[机器指纹]   9ad5e28c21bdc46e
[RUN_ID]     20244081132-9ad5e28c21bdc46e
[Python]     3.13.9


---

# 实验 1：正则表达式的基础匹配

导入 `re` 已在第一个代码框完成。本实验分三小题，分别练习**转义字符**（`\d`、`\s`、`\b`）、**择一匹配**（`|`）与**量词**（`*`、`+`、`?`、`{m}`、`{m,n}`）。

### 任务 1：`\d+`、`\s+`、`\bcat\b`

- `\d+`：匹配一个或多个数字；
- `\s+`：匹配一段连续空白（空格、制表、换行等）；
- `\bcat\b`：`\b` 是单词边界，只匹配**独立**的单词 `cat`，不会命中 `cats` 中的 `cat`。

> 注意：给定文本中没有数字，因此 `\d+` 的结果为空列表——这正说明正则只返回真实存在的匹配。

In [2]:
text = "cat on mat; later: cats and dogs"

print("原文本：", repr(text))
print("数字 \\d+        :", re.findall(r"\d+", text))
print("空白 \\s+        :", re.findall(r"\s+", text))
print("整词 cat \\bcat\\b:", re.findall(r"\bcat\b", text))

原文本： 'cat on mat; later: cats and dogs'
数字 \d+        : []
空白 \s+        : [' ', ' ', ' ', ' ', ' ', ' ']
整词 cat \bcat\b: ['cat']


**说明**：`\s+` 把每处连续空白合并为一个匹配项；`\bcat\b` 只匹配开头的独立 `cat`，`cats` 因后面紧跟字母 `s` 不构成单词边界而被排除。

### 任务 2：择一匹配 `cat|dog` 与交替顺序的影响

`A|B` 表示匹配 `A` 或 `B`。`re.findall` 会**从左到右**扫描文本，并在每个位置尝试**按模式中从左到右**列出的分支——因此分支顺序会改变结果：

- `a|aa|aaa`：先试 `a`，在 `"aaaa"` 的每个位置都立刻命中单字符 `a`，得到 4 个 `a`；
- `aaa|aa|a`：先试最长的 `aaa`，先吃掉 3 个 `a`，再在剩余位置匹配 1 个 `a`，得到 `['aaa', 'a']`。

In [3]:
text2 = "cat dog bird CAT"
print("cat|dog 匹配结果 :", re.findall(r"cat|dog", text2))

print("a|aa|aaa  on 'aaaa' :", re.findall(r"a|aa|aaa", "aaaa"))
print("aaa|aa|a  on 'aaaa' :", re.findall(r"aaa|aa|a", "aaaa"))

cat|dog 匹配结果 : ['cat', 'dog']
a|aa|aaa  on 'aaaa' : ['a', 'a', 'a', 'a']
aaa|aa|a  on 'aaaa' : ['aaa', 'a']


**说明**：`cat|dog` 默认区分大小写，故 `CAT` 不匹配；若需忽略大小写可传入 `re.IGNORECASE`。后两行结果不同，说明**把更长的分支写在前面**通常能得到"最长优先"的效果。

### 任务 3：量词 `*`、`+`、`?`、`{m}`、`{m,n}`

在字符串 `"ct cat caat caaat"` 上比较五种量词（均作用在字母 `a` 上）：

| 量词 | 含义 |
|---|---|
| `ca*t` | `a` 出现 **0 次或多次** |
| `ca+t` | `a` 出现 **1 次或多次** |
| `ca?t` | `a` 出现 **0 次或 1 次** |
| `ca{2}t` | `a` **恰好 2 次** |
| `ca{1,2}t` | `a` **1 到 2 次**（含两端） |

In [4]:
text3 = "ct cat caat caaat"
patterns = {
    "ca*t"     : r"ca*t",
    "ca+t"     : r"ca+t",
    "ca?t"     : r"ca?t",
    "ca{2}t"   : r"ca{2}t",
    "ca{1,2}t" : r"ca{1,2}t",
}
print("测试文本：", repr(text3))
for name, pat in patterns.items():
    print(f"{name:8s} -> {re.findall(pat, text3)}")

测试文本： 'ct cat caat caaat'
ca*t     -> ['ct', 'cat', 'caat', 'caaat']
ca+t     -> ['cat', 'caat', 'caaat']
ca?t     -> ['ct', 'cat']
ca{2}t   -> ['caat']
ca{1,2}t -> ['cat', 'caat']


**说明**：`ca*t` 连 `ct`（0 个 a）也匹配；`ca+t` 至少 1 个 a，排除 `ct`；`ca?t` 只允许 0 或 1 个 a，故只命中 `ct`、`cat`；`ca{2}t` 只命中 `caat`；`ca{1,2}t` 命中 `cat` 与 `caat`，排除 3 个 a 的 `caaat`。

---

# 实验 2：分组、回溯引用与替换

本实验练习**捕获组 `(...)`** 提取子串、**后向引用 `\1`** 查找重复内容，以及用 `re.sub()` 配合捕获组做结构化替换。

### 任务 1：用捕获组提取邮箱的用户名与域名

模式 `([\w.+-]+)@([\w-]+(?:\.[\w-]+)+)`：
- 第一个捕获组匹配 `@` 前的用户名（允许字母、数字、下划线、点、加号、连字符）；
- 第二个捕获组匹配域名（由一个或多个 `.` 分隔的段组成）。
用 `finditer()` 逐个取出 `group(1)` / `group(2)`。

In [5]:
contact = "联系：alice@example.com；备选：bob@test.org"
email_pat = r"([\w.+-]+)@([\w-]+(?:\.[\w-]+)+)"

emails = []
for m in re.finditer(email_pat, contact):
    emails.append((m.group(1), m.group(2)))
    print(f"完整邮箱={m.group(0):20s} 用户名={m.group(1):8s} 域名={m.group(2)}")

print("提取结果列表：", emails)

完整邮箱=alice@example.com    用户名=alice    域名=example.com
完整邮箱=bob@test.org         用户名=bob      域名=test.org
提取结果列表： [('alice', 'example.com'), ('bob', 'test.org')]


### 任务 2：用后向引用找出连续重复的单词

模式 `\b(\w+)\s+\1\b`：`(\w+)` 捕获一个单词，`\s+` 匹配其后的空白，`\1` 后向引用**同一个**捕获内容，`\b` 保证整词边界。文本中的 `cat cat`、`bird bird` 满足，`dog cat` 不满足。

In [6]:
dup_text = "cat cat, dog cat, bird bird"
dup_pat = r"\b(\w+)\s+\1\b"

dup_words = re.findall(dup_pat, dup_text)
print("原文本：", repr(dup_text))
print("连续重复的单词：", dup_words)
for w in dup_words:
    print(f"  重复词 -> {w}")

原文本： 'cat cat, dog cat, bird bird'
连续重复的单词： ['cat', 'bird']
  重复词 -> cat
  重复词 -> bird


### 任务 3：日期格式转换（年-月-日 → 月-日-年）

模式 `(\d{4})[-/.](\d{2})[-/.](\d{2})` 用三个捕获组分别取年、月、日，分隔符兼容 `-`、`/`、` .` 三种。替换为 `\2-\3-\1` 即"月-日-年"。用 `re.sub()` 一次性转换，**行数保持不变**（`\n` 未被改动）。

In [7]:
dates = "2024-05-01\n2024/05/02\n2024.05.03"
date_pat = r"(\d{4})[-/.](\d{2})[-/.](\d{2})"
converted = re.sub(date_pat, r"\2-\3-\1", dates)

print("转换前：")
print(dates)
print("\n转换后：")
print(converted)
print("\n行数   ：", len(dates.splitlines()), "->", len(converted.splitlines()))

转换前：
2024-05-01
2024/05/02
2024.05.03

转换后：
05-01-2024
05-02-2024
05-03-2024

行数   ： 3 -> 3


### 任务 4：统一电话号码格式

模式 `(\d{3})[-.](\d{3})[-.](\d{4})` 兼容 `-` 与 `.` 两种分隔符，替换为 `(\1) \2-\3`，得到统一的 `(123) 456-7890` 形式。

In [8]:
phone_text = "联系我：123-456-7890 或 987.654.3210"
phone_pat = r"(\d{3})[-.](\d{3})[-.](\d{4})"
phones = re.sub(phone_pat, r"(\1) \2-\3", phone_text)

print("原文本：", phone_text)
print("统一后：", phones)

原文本： 联系我：123-456-7890 或 987.654.3210
统一后： 联系我：(123) 456-7890 或 (987) 654-3210


---

# 实验 3：使用 BeautifulSoup 解析 HTML

本实验使用给定的 HTML 字符串（不依赖网页下载），用 `BeautifulSoup(html_doc, "html.parser")` 构建解析树，练习按标签名、`class` 属性查找节点，以及 `get_text()` 提取可见文本。

In [9]:
html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式</p>
<p class="course">第七讲：结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

soup = BeautifulSoup(html_doc, "html.parser")
print("解析树已构建，类型：", type(soup).__name__)

解析树已构建，类型： BeautifulSoup


### 任务 1：输出标题文本、h1 的标签名与文本

- `soup.title.string` / `soup.title.get_text()` 得到 `<title>` 的文本；
- `soup.h1.name` 得到标签名 `h1`，`soup.h1.get_text()` 得到其文本。

In [10]:
print("页面标题文本：", soup.title.get_text())
print("h1 标签名    ：", soup.h1.name)
print("h1 文本      ：", soup.h1.get_text())

页面标题文本： 实验课程
h1 标签名    ： h1
h1 文本      ： 数据分析


### 任务 2：用 find_all() 找 course 段落与所有链接

- `find_all("p", class_="course")` 找出所有 `class="course"` 的段落；
- `find_all("a")` 找出所有链接，逐项输出链接文字（`get_text()`）与 `href` 属性（`["href"]` 或 `get("href")`）。

In [11]:
course_ps = soup.find_all("p", class_="course")
print(f'class="course" 的段落共 {len(course_ps)} 个：')
for i, p in enumerate(course_ps, 1):
    print(f"  [{i}] {p.get_text()}")

links = soup.find_all("a")
print(f"\n链接共 {len(links)} 个：")
for i, a in enumerate(links, 1):
    print(f"  [{i}] 文字={a.get_text():6s} href={a.get('href')}")

class="course" 的段落共 2 个：
  [1] 第六讲：正则表达式
  [2] 第七讲：结构化数据

链接共 2 个：
  [1] 文字=讲义六    href=https://example.org/lecture06
  [2] 文字=讲义七    href=https://example.org/lecture07


### 任务 3：用 get_text() 提取可见文本并比较范围

`get_text()` 会递归拼接节点内所有文本，默认把不同节点文本直接相连；传入 `separator` 参数可插入分隔。下面比较**整个文档**、**单个段落**、**单个链接**三种范围的结果差异。

In [12]:
whole_text   = soup.get_text()
para_text    = course_ps[0].get_text()
link_text    = links[0].get_text()

print("整个文档 get_text()   :", repr(whole_text.strip()))
print("单个段落 get_text()   :", repr(para_text))
print("单个链接 get_text()   :", repr(link_text))
print("带分隔符 separator=' | ':", repr(soup.get_text(separator=" | ").strip()))

整个文档 get_text()   : '实验课程\n数据分析\n第六讲：正则表达式\n第七讲：结构化数据\n讲义六\n讲义七'
单个段落 get_text()   : '第六讲：正则表达式'
单个链接 get_text()   : '讲义六'
带分隔符 separator=' | ': '| 实验课程 | \n | 数据分析 | \n | 第六讲：正则表达式 | \n | 第七讲：结构化数据 | \n | 讲义六 | \n | 讲义七 | \n |'


**结论**：`get_text()` 的作用范围由调用它的节点决定——在 `soup` 上调用得到整页可见文字，在具体标签上调用只得到该标签子树内的文字；`separator` 让不同节点的文本更易读。

---

## 提交自检
- [x] 独立 conda 环境 `exp03`（非 base）
- [x] 第一个代码框为学号与环境记录，其后按顺序运行
- [x] 三个实验的关键输出均已显示，并配 Markdown 说明
- [x] 重启内核运行全部代码，输出已嵌入，无报错
- [x] 文件名：`20244081132_唐林_exp03.ipynb`